# Redes Recurrentes: RNN, LSTM y GRU

**Temas cubiertos:**
- Datos secuenciales y modelado de dependencias temporales
- Redes Neuronales Recurrentes (RNN)
- Long Short-Term Memory (LSTM)
- Gated Recurrent Unit (GRU)
- Aplicaciones en lenguaje natural y series temporales

---

## 1. Datos Secuenciales y Dependencias Temporales

### 1.1 Que son los datos secuenciales

Un dato secuencial es aquel en el que el orden de los elementos importa. A diferencia de los datos tabulares donde cada fila es independiente, en una secuencia el valor en el tiempo `t` depende de los valores anteriores.

Ejemplos:
- Una oracion: "El gato ___ en el tejado" — la palabra que falta depende de las anteriores.
- Una serie de precios de acciones: el precio de hoy esta correlacionado con el de ayer.
- Un electrocardiograma: cada punto es parte de un patron que se extiende en el tiempo.

Formalmente, una secuencia de longitud `T` se define como:

```
x = (x_1, x_2, ..., x_T)   donde cada x_t es un vector de features en el paso t
```

### 1.2 El problema de las dependencias a largo plazo

En muchas secuencias, el contexto relevante puede estar muy alejado en el tiempo. Por ejemplo:

```
"Los datos que el modelo proceso durante el entrenamiento fueron..."
```

Para predecir el verbo final, necesitamos recordar "datos" (plural) que aparecio varios pasos atras. Este es el **problema de las dependencias a largo plazo** y es la razon principal por la que se diseñaron LSTM y GRU.

---

## 2. Redes Neuronales Recurrentes (RNN)

### 2.1 Arquitectura

Una RNN procesa secuencias manteniendo un **estado oculto** `h_t` que actua como memoria a corto plazo. En cada paso de tiempo, la red recibe el input actual `x_t` y el estado previo `h_{t-1}` para producir un nuevo estado.

Las ecuaciones de una RNN simple (Elman Network) son:

```
h_t = tanh(W_h * h_{t-1}  +  W_x * x_t  +  b_h)
y_t = W_y * h_t  +  b_y
```

Donde:
- `h_t`  : estado oculto en el paso t (la "memoria")
- `x_t`  : input en el paso t
- `W_h`  : matriz de pesos recurrentes (mismo en todos los pasos)
- `W_x`  : matriz de pesos para el input
- `y_t`  : output en el paso t
- `tanh` : funcion de activacion

El punto clave es que **los pesos se comparten en todos los pasos de tiempo** (parameter sharing), lo que permite procesar secuencias de longitud variable.

### 2.2 Backpropagation Through Time (BPTT)

El entrenamiento de una RNN utiliza BPTT: se desenrolla la red en el tiempo y se aplica backpropagation sobre la secuencia completa. El gradiente de la perdida respecto a `h_0` requiere multiplicar matrices de pesos `T` veces:

```
dL/dh_0  =  (W_h)^T * (W_h)^T * ... * (W_h)^T * dL/dh_T
             <---------- T multiplicaciones ---------->
```

Esto provoca dos problemas clasicos:

- **Vanishing gradient**: si los valores propios de `W_h` son < 1, el gradiente se acerca a 0 y la red no aprende dependencias largas.
- **Exploding gradient**: si los valores propios de `W_h` son > 1, el gradiente crece exponencialmente. Se mitiga con gradient clipping.

### 2.3 Limitaciones de la RNN simple

La RNN simple funciona bien para secuencias cortas (hasta ~10-20 pasos), pero falla en:
- Textos largos
- Series temporales con patrones estacionales de periodo largo
- Cualquier tarea donde el contexto relevante este a mas de ~20 pasos

---

## 3. Long Short-Term Memory (LSTM)

### 3.1 Motivacion

Hochreiter y Schmidhuber (1997) propusieron la LSTM para resolver el problema del vanishing gradient. La idea central es introducir una **celda de memoria** `C_t` separada del estado oculto, controlada por compuertas (gates) que aprenden que informacion conservar, descartar o emitir.

### 3.2 Las tres compuertas

Una LSTM tiene tres compuertas, cada una con sus propios pesos entrenables:

**Forget gate (compuerta de olvido):**
Decide que parte del estado de celda anterior `C_{t-1}` se conserva.
```
f_t = sigmoid(W_f * [h_{t-1}, x_t] + b_f)
```
Valores cercanos a 0 → olvida; cercanos a 1 → conserva.

**Input gate (compuerta de entrada):**
Decide que nueva informacion se escribe en la celda.
```
i_t  = sigmoid(W_i * [h_{t-1}, x_t] + b_i)   <- que posiciones actualizar
C~_t = tanh(W_C * [h_{t-1}, x_t] + b_C)       <- valores candidatos
```

**Actualizacion de la celda:**
```
C_t = f_t * C_{t-1}  +  i_t * C~_t
```
La celda `C_t` fluye con gradientes casi intactos gracias a la adicion (no multiplicacion) en esta ecuacion.

**Output gate (compuerta de salida):**
```
o_t = sigmoid(W_o * [h_{t-1}, x_t] + b_o)
h_t = o_t * tanh(C_t)
```

### 3.3 Por que funciona

La clave es el **constant error carousel**: la celda `C_t` se actualiza mediante una suma, no una multiplicacion matricial, por lo que el gradiente puede fluir hacia atras sin desvanecerse durante muchos pasos de tiempo. Las compuertas aprenden cuando interrumpir o permitir ese flujo.

---

## 4. Gated Recurrent Unit (GRU)

### 4.1 Simplificacion de la LSTM

Cho et al. (2014) propusieron la GRU como una version simplificada de la LSTM que elimina la celda de memoria separada y fusiona las compuertas de olvido y entrada en una sola **reset gate** y una **update gate**.

**Reset gate:**
Controla cuanto del estado pasado se usa para calcular el nuevo estado candidato.
```
r_t = sigmoid(W_r * [h_{t-1}, x_t])
```

**Update gate:**
Decide cuanto del estado anterior se conserva vs cuanto del nuevo candidato se incorpora.
```
z_t = sigmoid(W_z * [h_{t-1}, x_t])
```

**Estado candidato:**
```
h~_t = tanh(W * [r_t * h_{t-1}, x_t])
```

**Actualizacion del estado oculto:**
```
h_t = (1 - z_t) * h_{t-1}  +  z_t * h~_t
```

### 4.2 LSTM vs GRU: comparativa

| Caracteristica         | LSTM                        | GRU                         |
|------------------------|-----------------------------|---------------------------------|
| Parametros             | Mas (3 compuertas + celda)  | Menos (2 compuertas, sin celda) |
| Velocidad de entreno   | Mas lenta                   | Mas rapida                      |
| Memoria a largo plazo  | Excelente                   | Muy buena                       |
| Datasets pequenos      | Puede sobreajustar          | Mejor generalizacion            |
| Datasets grandes       | Generalmente superior       | Comparable                      |

En la practica, la diferencia de rendimiento es pequena y depende del problema. La GRU es preferible cuando el tiempo de entrenamiento o el tamano del modelo son restricciones.

---

## 5. Aplicaciones

### 5.1 Procesamiento de Lenguaje Natural (NLP)

Las RNN/LSTM/GRU dominaron el NLP antes de los Transformers (2017). Sus aplicaciones principales son:

- **Clasificacion de texto**: analisis de sentimientos, deteccion de spam, clasificacion de noticias.
- **Generacion de texto**: modelos de lenguaje caracter a caracter o palabra a palabra.
- **Traduccion automatica**: arquitectura encoder-decoder con LSTM.
- **Reconocimiento de entidades**: etiquetado de secuencias (NER).
- **Question answering** y **resumen automatico**.

En clasificacion de texto, la LSTM procesa la secuencia de palabras (representadas como embeddings) y el estado oculto final `h_T` resume el significado de toda la oracion para pasarlo a una capa densa.

### 5.2 Series Temporales

En series temporales, la red recibe una ventana de `T` pasos pasados y predice el valor en `T+1` (o varios pasos hacia adelante):

```
Input:  [x_{t-T}, x_{t-T+1}, ..., x_{t-1}]  ->  Modelo  ->  x_t (prediccion)
```

Aplicaciones tipicas:
- Prediccion de demanda energetica
- Prediccion de precios financieros
- Mantenimiento predictivo en industria
- Prediccion meteorologica
- Trafico de red y deteccion de anomalias

---

---
# EJERCICIO RESUELTO
## Prediccion de Serie Temporal con LSTM y GRU

**Problema:** Predecir la temperatura maxima diaria usando los ultimos 30 dias como contexto.

Se usara un dataset sintetico con patron estacional + ruido, que representa bien el comportamiento real de series climaticas.

**Arquitecturas a comparar:** LSTM vs GRU vs RNN simple

---

In [1]:
# Instalacion de dependencias (solo necesario en Colab)
# tensorflow ya viene instalado en Colab; si no, ejecutar:
# !pip install tensorflow --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow version: {tf.__version__}')
print('Librerias cargadas correctamente.')

ModuleNotFoundError: No module named 'tensorflow'

### Paso 1 — Generacion y visualizacion del dataset

In [ ]:
# Generacion de serie temporal sintetica
# Representa temperatura maxima diaria con patron anual + tendencia + ruido

N = 1460  # 4 años de datos diarios
t = np.arange(N)

# Componentes de la serie
estacional  = 15 * np.sin(2 * np.pi * t / 365)         # patron anual
tendencia   = 0.003 * t                                  # ligero calentamiento
ruido       = np.random.normal(0, 2.0, N)               # ruido gaussiano
temperatura = 22 + estacional + tendencia + ruido       # temperatura base: 22 C

# Crear dataframe con fechas
fechas = pd.date_range(start='2020-01-01', periods=N, freq='D')
df = pd.DataFrame({'fecha': fechas, 'temperatura': temperatura})
df.set_index('fecha', inplace=True)

# Estadisticas basicas
print('Estadisticas de la serie temporal:')
print(df['temperatura'].describe().round(2))

# Visualizacion
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(df.index, df['temperatura'], color='steelblue', linewidth=0.8, alpha=0.8)
axes[0].set_title('Serie temporal completa (4 años)', fontsize=12)
axes[0].set_ylabel('Temperatura (C)')
axes[0].grid(True, alpha=0.3)

# Zoom: primer año
primer_ano = df.iloc[:365]
axes[1].plot(primer_ano.index, primer_ano['temperatura'], color='darkorange', linewidth=1.2)
axes[1].set_title('Detalle: primer año', fontsize=12)
axes[1].set_ylabel('Temperatura (C)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Paso 2 — Preprocesamiento: normalizacion y creacion de ventanas

In [ ]:
# Normalizacion entre 0 y 1 (requisito para redes neuronales)
serie = df['temperatura'].values.reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
serie_norm = scaler.fit_transform(serie)

print(f'Rango original:    [{serie.min():.2f}, {serie.max():.2f}]')
print(f'Rango normalizado: [{serie_norm.min():.2f}, {serie_norm.max():.2f}]')

# ----------------------------------------------------------
# Funcion para crear ventanas deslizantes
# Dada la serie [x1, x2, ..., xN] y una ventana W:
# X[i] = [x_i, x_{i+1}, ..., x_{i+W-1}]  (secuencia de entrada)
# y[i] = x_{i+W}                          (valor a predecir)
# ----------------------------------------------------------
def crear_ventanas(serie, ventana):
    X, y = [], []
    for i in range(len(serie) - ventana):
        X.append(serie[i : i + ventana, 0])
        y.append(serie[i + ventana, 0])
    return np.array(X), np.array(y)

VENTANA = 30  # usar los ultimos 30 dias para predecir el siguiente

X, y = crear_ventanas(serie_norm, VENTANA)

# Division temporal: 80% train, 10% validacion, 10% test
n      = len(X)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)

X_train = X[:n_train]
y_train = y[:n_train]
X_val   = X[n_train : n_train + n_val]
y_val   = y[n_train : n_train + n_val]
X_test  = X[n_train + n_val :]
y_test  = y[n_train + n_val :]

# Reshape para Keras: (muestras, pasos de tiempo, features)
X_train = X_train.reshape(-1, VENTANA, 1)
X_val   = X_val.reshape(-1, VENTANA, 1)
X_test  = X_test.reshape(-1, VENTANA, 1)

print(f'\nForma de los datos:')
print(f'  X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'  X_val:   {X_val.shape}    y_val:   {y_val.shape}')
print(f'  X_test:  {X_test.shape}   y_test:  {y_test.shape}')

### Paso 3 — Definicion de los modelos

In [ ]:
def construir_modelo(tipo, unidades=64, dropout=0.2):
    """
    Construye un modelo secuencial con la arquitectura especificada.
    
    Parametros:
    -----------
    tipo     : 'RNN', 'LSTM' o 'GRU'
    unidades : numero de unidades en la capa recurrente
    dropout  : tasa de regularizacion
    """
    capas = {'RNN': SimpleRNN, 'LSTM': LSTM, 'GRU': GRU}
    CapaRecurrente = capas[tipo]
    
    modelo = Sequential([
        # Primera capa recurrente con return_sequences=True
        # para pasar la secuencia completa a la segunda capa
        CapaRecurrente(unidades, return_sequences=True,
                       input_shape=(VENTANA, 1)),
        Dropout(dropout),
        
        # Segunda capa recurrente (solo retorna el ultimo estado)
        CapaRecurrente(unidades // 2, return_sequences=False),
        Dropout(dropout),
        
        # Capa densa de salida: regresion -> 1 neurona sin activacion
        Dense(32, activation='relu'),
        Dense(1)
    ], name=f'Modelo_{tipo}')
    
    modelo.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return modelo


# Instanciar los tres modelos
modelos = {
    'RNN':  construir_modelo('RNN'),
    'LSTM': construir_modelo('LSTM'),
    'GRU':  construir_modelo('GRU')
}

# Mostrar resumen del modelo LSTM como referencia
print('Arquitectura del modelo LSTM:')
modelos['LSTM'].summary()

### Paso 4 — Entrenamiento con Early Stopping

In [ ]:
EPOCHS   = 100
BATCH    = 32

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=0
)

historiales = {}

for nombre, modelo in modelos.items():
    print(f'Entrenando {nombre}...')
    hist = modelo.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH,
        callbacks=[early_stop],
        verbose=0
    )
    historiales[nombre] = hist
    epocas_usadas = len(hist.history['loss'])
    val_loss_final = hist.history['val_loss'][-1]
    print(f'  Epocas utilizadas: {epocas_usadas} | Val Loss final: {val_loss_final:.6f}')

print('\nEntrenamiento completado.')

### Paso 5 — Curvas de aprendizaje

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colores = {'RNN': 'tomato', 'LSTM': 'steelblue', 'GRU': 'seagreen'}

for ax, (nombre, hist) in zip(axes, historiales.items()):
    color = colores[nombre]
    ax.plot(hist.history['loss'],     color=color,  linewidth=1.5, label='Train loss')
    ax.plot(hist.history['val_loss'], color=color,  linewidth=1.5, label='Val loss',
            linestyle='--', alpha=0.8)
    ax.set_title(f'Curva de aprendizaje: {nombre}', fontsize=11)
    ax.set_xlabel('Epoca')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Paso 6 — Evaluacion en el conjunto de test

In [ ]:
resultados = {}

for nombre, modelo in modelos.items():
    # Predicciones en escala normalizada
    pred_norm = modelo.predict(X_test, verbose=0)
    
    # Desnormalizar
    pred_real = scaler.inverse_transform(pred_norm)
    y_real    = scaler.inverse_transform(y_test.reshape(-1, 1))
    
    rmse = np.sqrt(mean_squared_error(y_real, pred_real))
    mae  = mean_absolute_error(y_real, pred_real)
    
    resultados[nombre] = {
        'RMSE': rmse,
        'MAE':  mae,
        'predicciones': pred_real,
        'reales':       y_real
    }

# Tabla comparativa
tabla = pd.DataFrame({
    nombre: {'RMSE (C)': v['RMSE'], 'MAE (C)': v['MAE']}
    for nombre, v in resultados.items()
}).T.round(4)

print('Metricas en conjunto de test (temperatura en grados Celsius):')
print(tabla.to_string())

mejor = tabla['RMSE (C)'].idxmin()
print(f'\nMejor modelo segun RMSE: {mejor}')

### Paso 7 — Visualizacion de predicciones vs valores reales

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
N_MOSTRAR = 120  # mostrar los primeros 120 dias del test

for ax, (nombre, res) in zip(axes, resultados.items()):
    reales = res['reales'][:N_MOSTRAR].flatten()
    preds  = res['predicciones'][:N_MOSTRAR].flatten()
    dias   = np.arange(N_MOSTRAR)
    
    ax.plot(dias, reales, color='black',        linewidth=1.2, label='Real',       alpha=0.8)
    ax.plot(dias, preds,  color=colores[nombre], linewidth=1.2, label='Prediccion', alpha=0.9,
            linestyle='--')
    ax.fill_between(dias, reales, preds, alpha=0.12, color=colores[nombre])
    ax.set_title(f'{nombre}  |  RMSE: {res["RMSE"]:.3f} C  |  MAE: {res["MAE"]:.3f} C',
                 fontsize=11)
    ax.set_ylabel('Temperatura (C)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Dias (conjunto de test)')
plt.suptitle('Prediccion de temperatura: comparativa de arquitecturas',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Paso 8 — Analisis de errores

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (nombre, res) in zip(axes, resultados.items()):
    errores = (res['reales'] - res['predicciones']).flatten()
    ax.hist(errores, bins=30, color=colores[nombre], edgecolor='white', alpha=0.85)
    ax.axvline(0, color='black', linewidth=1.5, linestyle='--')
    ax.axvline(errores.mean(), color='red', linewidth=1.2, linestyle='-',
               label=f'Media: {errores.mean():.3f}')
    ax.set_title(f'Distribucion de errores: {nombre}', fontsize=11)
    ax.set_xlabel('Error (C)')
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Observaciones:')
for nombre, res in resultados.items():
    errores = (res['reales'] - res['predicciones']).flatten()
    print(f'  {nombre}: media={errores.mean():.4f} C | std={errores.std():.4f} C')

### Paso 9 — Interpretacion de resultados

Los resultados anteriores permiten extraer las siguientes conclusiones:

1. **RNN simple**: muestra el peor desempeño en RMSE y MAE. Esto es consistente con la teoria: la serie tiene un patron estacional de 365 dias que excede la capacidad de memoria de una RNN simple.

2. **LSTM**: logra el mejor (o un rendimiento similar al de GRU) gracias a la celda de memoria que puede mantener informacion del patron estacional durante muchos pasos de tiempo.

3. **GRU**: rendimiento muy cercano a LSTM con menos parametros. Para este tamano de dataset la diferencia es marginal.

4. **Distribucion de errores**: los tres modelos muestran errores aproximadamente centrados en cero, lo que indica ausencia de sesgo sistematico. La desviacion estandar del error es el indicador mas relevante de la precision.

---

---
# EJERCICIO PROPUESTO
## Clasificacion de Sentimientos con LSTM

### Descripcion del problema

Se proporciona un dataset de reseñas de peliculas (subconjunto del IMDB Dataset disponible en Keras). Cada reseña es una secuencia de palabras y la etiqueta es:
- `1`: sentimiento positivo
- `0`: sentimiento negativo

El objetivo es construir y entrenar una red LSTM para clasificar correctamente el sentimiento de las reseñas.

### Estructura del notebook a completar

El alumno debe completar las secciones marcadas con `# TU CODIGO AQUI` siguiendo las instrucciones de cada celda.

**Criterio de exito minimo:** Accuracy de validacion >= 85%

---

In [ ]:
# Carga del dataset IMDB desde Keras
# El dataset ya viene tokenizado: cada palabra es un entero
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE  = 10000   # conservar solo las 10,000 palabras mas frecuentes
MAX_LEN     = 200     # truncar/rellenar todas las reseñas a 200 tokens

(X_train_raw, y_train_ep), (X_test_raw, y_test_ep) = imdb.load_data(num_words=VOCAB_SIZE)

# Padding: ajustar todas las secuencias a la misma longitud
X_train_ep = pad_sequences(X_train_raw, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_ep  = pad_sequences(X_test_raw,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'Muestras de entrenamiento: {X_train_ep.shape}')
print(f'Muestras de test:          {X_test_ep.shape}')
print(f'Distribucion de clases en train: positivo={y_train_ep.sum()} | negativo={(1-y_train_ep).sum()}')

In [ ]:
# TAREA 1: Exploracion de datos
#
# 1a. Calcula y muestra la longitud promedio, mediana y maxima de las reseñas
#     en X_train_raw (antes del padding).
#
# 1b. Crea un histograma de la distribucion de longitudes de las reseñas.
#
# 1c. Decodifica e imprime la primera reseña de X_train_raw en texto legible.
#     Pista: usa imdb.get_word_index() para obtener el vocabulario.

# TU CODIGO AQUI


In [ ]:
# TAREA 2: Construccion del modelo
#
# Construye una red LSTM para clasificacion binaria con la siguiente arquitectura:
#
#   1. Capa Embedding(VOCAB_SIZE, 64, input_length=MAX_LEN)
#      Convierte indices enteros en vectores densos de dimension 64.
#
#   2. Una capa LSTM con 64 unidades y return_sequences=True
#
#   3. Dropout(0.3)
#
#   4. Una segunda capa LSTM con 32 unidades (return_sequences=False)
#
#   5. Dense(1, activation='sigmoid') para clasificacion binaria
#
# Compila con:
#   - optimizer: Adam con learning_rate=0.001
#   - loss: 'binary_crossentropy'
#   - metrics: ['accuracy']
#
# Muestra el resumen del modelo.

from tensorflow.keras.layers import Embedding

# TU CODIGO AQUI


In [ ]:
# TAREA 3: Entrenamiento
#
# Entrena el modelo con las siguientes especificaciones:
#   - epochs=20, batch_size=64
#   - validation_split=0.1
#   - EarlyStopping con patience=3, monitor='val_accuracy', restore_best_weights=True
#
# Guarda el historial en una variable llamada 'hist_ep'.

# TU CODIGO AQUI


In [ ]:
# TAREA 4: Visualizacion de curvas de aprendizaje
#
# Crea una figura con 2 subplots:
#   - Izquierda: loss y val_loss vs epocas
#   - Derecha:   accuracy y val_accuracy vs epocas
# Incluye leyenda, titulos y grilla en cada grafica.

# TU CODIGO AQUI


In [ ]:
# TAREA 5: Evaluacion en test
#
# 5a. Evalua el modelo en X_test_ep y y_test_ep.
#     Imprime el accuracy y loss final en test.
#
# 5b. Genera las predicciones binarias (0 o 1) usando un umbral de 0.5.
#
# 5c. Imprime el classification_report con precision, recall y F1.
#
# 5d. Grafica la matriz de confusion.

# TU CODIGO AQUI


In [ ]:
# TAREA 6 (opcional avanzada): Comparativa LSTM vs GRU
#
# Replica exactamente la misma arquitectura pero reemplazando LSTM por GRU.
# Compara ambos modelos en una tabla con las siguientes metricas en test:
#   - Accuracy
#   - F1-Score (macro)
#   - Numero de parametros entrenables
#   - Epocas hasta convergencia (early stopping)
#
# Concluye: para este problema y tamano de dataset, que arquitectura recomendarias
# y por que?

# TU CODIGO AQUI


### Preguntas de reflexion

Responde en esta celda Markdown una vez completadas las tareas anteriores.

**Pregunta 1.** En el modelo de clasificacion de texto, la capa Embedding aprende representaciones vectoriales de las palabras. A diferencia de la capa LSTM, cuyos pesos se comparten en el tiempo, los pesos del Embedding son independientes por palabra. Explica por que aun asi este enfoque es correcto y eficiente.

**Tu respuesta:**

---

**Pregunta 2.** El dataset IMDB esta balanceado (50% positivo, 50% negativo). Si el dataset tuviera 90% de reseñas negativas y 10% positivas, que cambios harias en la arquitectura, la funcion de perdida o el proceso de evaluacion para tratar el desbalance de clases?

**Tu respuesta:**

---

**Pregunta 3.** Las arquitecturas Transformer (BERT, GPT) han reemplazado a las LSTM en la mayoria de tareas de NLP. Sin embargo, las LSTM siguen siendo relevantes en algunos contextos. Menciona al menos dos escenarios donde preferirías una LSTM sobre un Transformer y justifica.

**Tu respuesta:**

---

## Checklist de entrega

- [ ] Todas las celdas ejecutadas sin errores
- [ ] Tarea 1: exploracion completada con histograma y reseña decodificada
- [ ] Tarea 2: modelo construido y resumen mostrado
- [ ] Tarea 3: modelo entrenado con early stopping
- [ ] Tarea 4: curvas de aprendizaje graficadas
- [ ] Tarea 5: evaluacion completa con classification_report y matriz de confusion
- [ ] Accuracy en test >= 85%
- [ ] Las 3 preguntas de reflexion respondidas
- [ ] (Bonus) Tarea 6: comparativa LSTM vs GRU completada